# Baseline OCR Evaluation

## Candidate 001

This notebook evaluates NARA's auto-generated OCR for Candidate 001
against the independently created and locked OCR evaluation reference.

The raw NARA OCR is preserved unchanged.

A derived evaluation-aligned OCR representation is created only to match
the textual scope defined in the frozen Reference Transcription Protocol.
No OCR errors are corrected during alignment.

In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent

REFERENCE_PATH = (
    REPO_ROOT
    / "data"
    / "ground_truth"
    / "candidate_001_ocr_reference.txt"
)

RAW_OCR_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "candidate_001_nara_ocr.txt"
)

In [2]:
reference_text = REFERENCE_PATH.read_text(encoding="utf-8")
raw_ocr_text = RAW_OCR_PATH.read_text(encoding="utf-8")

print("Reference characters:", len(reference_text))
print("Raw OCR characters:", len(raw_ocr_text))

Reference characters: 2046
Raw OCR characters: 2175


## Evaluation Alignment Rules

The raw NARA OCR is not edited or overwritten.

For baseline comparison, the derived OCR representation follows the same
scope rules used to create the locked OCR evaluation reference:

- exclude later declassification text;
- exclude margin annotations such as `TAH`;
- exclude handwritten insertions;
- exclude handwritten/signature artifacts;
- retain the typed `DR. SANTE`;
- collapse whitespace introduced by physical line wrapping;
- join words divided only by end-of-line hyphenation;
- preserve all OCR recognition errors exactly as produced.

No spelling, names, punctuation, or apparent OCR errors are corrected.

In [4]:
# Preserve the raw OCR unchanged.
# Create a separate working copy for evaluation alignment.

ocr_lines = raw_ocr_text.splitlines()

EXCLUDED_LINES = {
    "Declassified per Executive Order 12958, Section 3.5",
    "NND Project Number: NND 775057 By: NND Date: 1977",
    "TAH",
    "presimed",
    "Powin",
}

aligned_lines = [
    line
    for line in ocr_lines
    if line.strip() not in EXCLUDED_LINES
]

aligned_ocr_text = "\n".join(aligned_lines)

print(aligned_ocr_text)

OFFICE OF MILITART GOVERNMENT
FOR GREATER HESSE
ECONOMICS DIVISION
7 March 1947
MEMORANDUM TO Mr. Heinrich, Chief MFA&A Section
SUBJECT: Legal ownership of archives and libraries
1. In stating legal ownership of archives and libraries there is a
difference between the big and the small institutions.
2. All big institutions have got the questionnaire for G/erman archival
institutions and libraries and filled out the questionsaòf "acquisitions sub-
sequent to 1 Jan 1938," of confiscated materials. The most institutions, the
Statearchives at Darmstadt and Marburg, the National archives and the City ar-
chives at Frankfurt, the LandMibraries at Darmstadt and Kassel, the University-
libraries at Darmsbadt, Frankfurt and Giessen have reported no confisvated ma-
terials. The Statearchivesat Wiesbaden had got documents exchanged with Nether-
landsmarchives. They have already been returned to the Hague. The Library at
Fulda had reported several Jewish books, they have been fetched off. the book

In [5]:
print("Raw OCR characters:", len(raw_ocr_text))
print("Aligned OCR characters:", len(aligned_ocr_text))
print("Reference characters:", len(reference_text))

Raw OCR characters: 2175
Aligned OCR characters: 2054
Reference characters: 2046


In [6]:
import re

def normalize_for_evaluation(text):
    # Normalize line endings first.
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Known layout-only line break:
    # source has "University-" at line end followed by "libraries"
    # but the intended text is two separate words.
    text = re.sub(
        r"University-\s*\n\s*libraries",
        "University libraries",
        text
    )

    # Join words split only by end-of-line hyphenation:
    # sub-\nsequent -> subsequent
    # ar-\nchives -> archives
    # commu-\nnities -> communities
    text = re.sub(r"-\s*\n\s*", "", text)

    # Collapse remaining whitespace and physical line breaks.
    text = re.sub(r"\s+", " ", text).strip()

    return text


normalized_reference = normalize_for_evaluation(reference_text)
normalized_ocr = normalize_for_evaluation(aligned_ocr_text)

print("Normalized reference characters:", len(normalized_reference))
print("Normalized OCR characters:", len(normalized_ocr))

Normalized reference characters: 2039
Normalized OCR characters: 2039


In [7]:
print(normalized_reference[:500])
print()
print(normalized_ocr[:500])

OFFICE OF MILITART GOVERNMENT FOR GREATER HESSE ECONOMICS DIVISION 7 March 1947 MEMORANDUM TO Mr. Heinrich, Chief MFA&A Section SUBJECT: Legal ownership of archives and libraries 1. In stating legal ownership of archives and libraries there is a difference between the big and the small institutions. 2. All big institutions have got the questionnaire for German archival institutions and libraries and filled out the questions of "aquisitions subsequent to 1 Jan 1938," of confiscated materials. The

OFFICE OF MILITART GOVERNMENT FOR GREATER HESSE ECONOMICS DIVISION 7 March 1947 MEMORANDUM TO Mr. Heinrich, Chief MFA&A Section SUBJECT: Legal ownership of archives and libraries 1. In stating legal ownership of archives and libraries there is a difference between the big and the small institutions. 2. All big institutions have got the questionnaire for G/erman archival institutions and libraries and filled out the questionsaòf "acquisitions subsequent to 1 Jan 1938," of confiscated materials.

In [9]:
def levenshtein_distance(seq1, seq2):
    previous_row = list(range(len(seq2) + 1))

    for i, item1 in enumerate(seq1, start=1):
        current_row = [i]

        for j, item2 in enumerate(seq2, start=1):
            insert_cost = current_row[j - 1] + 1
            delete_cost = previous_row[j] + 1
            substitute_cost = previous_row[j - 1] + (item1 != item2)

            current_row.append(
                min(insert_cost, delete_cost, substitute_cost)
            )

        previous_row = current_row

    return previous_row[-1]

In [10]:
char_edits = levenshtein_distance(
    normalized_reference,
    normalized_ocr
)

cer = char_edits / len(normalized_reference)

reference_words = normalized_reference.split()
ocr_words = normalized_ocr.split()

word_edits = levenshtein_distance(
    reference_words,
    ocr_words
)

wer = word_edits / len(reference_words)

print("Character edits:", char_edits)
print(f"CER: {cer:.4%}")
print("Word edits:", word_edits)
print(f"WER: {wer:.4%}")

Character edits: 16
CER: 0.7847%
Word edits: 19
WER: 6.0703%


## Baseline Results — Candidate 001

After evaluation-scope alignment and layout normalization:

- Reference characters: 2,039
- Character edit distance: 16
- Character Error Rate (CER): **0.7847%**
- Word edit distance: 19
- Word Error Rate (WER): **6.0703%**

Although overall character-level recognition is high, several observed
errors occur within historically meaningful terms, including names of
institutions, places, and roles. These errors will be evaluated separately
for their effects on entity and relationship preservation.

These results describe Candidate 001 only and are not treated as
corpus-level estimates.

In [11]:
ALIGNED_OCR_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "candidate_001_nara_ocr_evaluation_aligned.txt"
)

ALIGNED_OCR_PATH.write_text(
    aligned_ocr_text,
    encoding="utf-8"
)

print(ALIGNED_OCR_PATH)

/Users/ab/Projects/misreading-the-archive/data/processed/candidate_001_nara_ocr_evaluation_aligned.txt
